In [1]:
import os
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
import numpy as np
import evaluate
# prevent annoying warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

/Users/jonmay/miniconda3/envs/openai2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [3]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)


In [4]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [5]:
# full bert is about 2x slower 
model_name="distilbert/distilbert-base-cased"#distilbert/distilbert-base-uncased"

In [6]:
# dataset -- has text and label divided into "train" and "test" segments
imdb = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained(model_name)


/Users/jonmay/miniconda3/envs/openai2/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
# collator -- handles batching
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")
train_dataset=imdb["train"].shuffle(seed=42).select(range(1000))
eval_dataset=imdb["test"].shuffle(seed=42).select(range(100))


In [8]:
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2)
training_args = TrainingArguments(
    output_dir="test",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="steps", # can also do "epoch" 
    eval_steps=20,
    save_strategy="steps",
    load_best_model_at_end=True,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [10]:
trainer.evaluate(tokenized_eval_dataset)


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jonmay (cutelabname). Use `wandb login --relogin` to force relogin


{'eval_loss': 0.7049955725669861,
 'eval_model_preparation_time': 0.0013,
 'eval_accuracy': 0.47,
 'eval_runtime': 5.2274,
 'eval_samples_per_second': 19.13,
 'eval_steps_per_second': 1.339}

In [11]:
trainer.train()


Step,Training Loss,Validation Loss,Model Preparation Time,Accuracy
20,No log,0.668984,0.001300,0.540000
40,No log,0.532485,0.001300,0.780000
60,No log,0.393085,0.001300,0.820000
80,No log,0.389207,0.001300,0.810000
100,No log,0.386241,0.001300,0.850000
120,No log,0.366034,0.001300,0.830000


TrainOutput(global_step=126, training_loss=0.4433224390423487, metrics={'train_runtime': 134.8592, 'train_samples_per_second': 14.83, 'train_steps_per_second': 0.934, 'total_flos': 262556593545504.0, 'train_loss': 0.4433224390423487, 'epoch': 2.0})

In [12]:
trainer.evaluate(tokenized_eval_dataset)


{'eval_loss': 0.3686765730381012,
 'eval_model_preparation_time': 0.0013,
 'eval_accuracy': 0.83,
 'eval_runtime': 1.9367,
 'eval_samples_per_second': 51.635,
 'eval_steps_per_second': 3.614,
 'epoch': 2.0}

In [13]:
eval_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 100
})

In [14]:
dir(eval_dataset)

['_TF_DATASET_REFS',
 '__class__',
 '__del__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getitems__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_build_local_temp_path',
 '_check_index_is_initialized',
 '_data',
 '_estimate_nbytes',
 '_fingerprint',
 '_format_columns',
 '_format_kwargs',
 '_format_type',
 '_generate_tables_from_cache_file',
 '_generate_tables_from_shards',
 '_get_cache_file_path',
 '_get_output_signature',
 '_getitem',
 '_indexes',
 '_indices',
 '_info',
 '_map_single',
 '_new_dataset_with_indices',
 '_output_all_columns',
 '_push_parquet_shards_to_hub',
 '_save_to_disk_single',
 '_select_contigu

In [23]:
eval_dataset.to_pandas().loc[0, 'label']

1